# Assignment 1A — Part B
Create grounded instruction pairs, train QLoRA adapters A/B/C, and compare the same three banking prompts.

### 1. Clone the repository

In [1]:
from pathlib import Path
import subprocess
import os

REPO_URL = "https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git"
PROJECT = Path("/content/banking-compliance-llm-assignment-1a")

if not (PROJECT / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)

### Install dependencies:

In [2]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00


### 2. Mount Drive and restore Part A files

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path
import shutil

SAVE_DIR = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_PART_A_RESULTS")

items = [
    "data/cleaned_txt",
    "data/train_corpus",
    "data/eval_corpus",
    "data/train_packed.parquet",
    "reports",
    "results",
    "models/cpt_model",
]

for item in items:
    source = SAVE_DIR / item
    destination = PROJECT / item

    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

print("Part A files restored.")

Part A files restored.


### Verify:

In [6]:
!ls models/cpt_model
!find data/cleaned_txt -type f -iname "*.txt" | wc -l

checkpoint-400	generation_config.json	tokenizer.json
checkpoint-500	model.safetensors	training_args.bin
config.json	tokenizer_config.json
118


### 3. Create the instruction dataset

In [7]:
!python -m src.data.create_instruction_dataset --minimum 100

2026-09-01 06:00:31,767 | INFO | create_instruction_dataset | Created 179824 pairs (143859 train / 35965 eval)


### Verify:

In [8]:
import json

print(json.dumps(
    json.loads(
        Path("data/instruction_dataset/split_report.json").read_text()
    ),
    indent=2
))

{
  "total_pairs": 179824,
  "train_pairs": 143859,
  "eval_pairs": 35965,
  "train_fraction": 0.7999988878014058,
  "seed": 42,
  "generation_method": "deterministic heuristic sentence grounding"
}


### 4. Train adapters A, B, and C

In [22]:
from pathlib import Path
import json

project = Path("/content/banking-compliance-llm-assignment-1a")
source = project / "data/instruction_dataset/train.jsonl"
clean_file = project / "data/instruction_dataset/train_for_adapter.jsonl"

valid_rows = []
bad_lines = []

with source.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, 1):
        if not line.strip():
            continue

        try:
            valid_rows.append(json.loads(line))
        except json.JSONDecodeError:
            bad_lines.append(line_number)

# The assignment requires at least 100 pairs.
# 1,000 is sufficient and much easier for Colab to handle.
valid_rows = valid_rows[:1000]

with clean_file.open("w", encoding="utf-8") as file:
    for row in valid_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Created:", clean_file)
print("Valid records:", len(valid_rows))
print("Skipped malformed lines:", len(bad_lines))

Created: /content/banking-compliance-llm-assignment-1a/data/instruction_dataset/train_for_adapter.jsonl
Valid records: 1000
Skipped malformed lines: 0


In [23]:
import json

with open(
    "/content/banking-compliance-llm-assignment-1a/data/instruction_dataset/train_for_adapter.jsonl",
    encoding="utf-8"
) as file:
    rows = [json.loads(line) for line in file if line.strip()]

print("Validated records:", len(rows))

Validated records: 1000


In [24]:
%cd /content/banking-compliance-llm-assignment-1a

!python -m src.qlora.train_adapter_A \
  --dataset /content/banking-compliance-llm-assignment-1a/data/instruction_dataset/train_for_adapter.jsonl \
  --max-steps 500

/content
2026-09-01 07:01:00,995 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
2026-09-01 07:01:01,475 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-09-01 07:01:01,476 | INFO | datasets | JAX version 0.11.1 available.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 656.02it/s]
Adding EOS to train dataset: 100% 1000/1000 [00:00<00:00, 27412.14 examples/s]
Tokenizing train dataset: 100% 1000/1000 [00:00<00:00, 1818.25 examples/s]
Building labels for train dataset: 100% 1000/1000 [00:00<00:00, 6580.10 examples/s]
Truncating train dataset: 100% 1000/1000 [00:00<00:00, 10281.62 examples/s]
Dropping fully masked examples from train dataset: 100% 1000/1000 [00:00<00:00, 51108.29 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's value

In [25]:
!python -m src.qlora.train_adapter_B \
  --dataset /content/banking-compliance-llm-assignment-1a/data/instruction_dataset/train_for_adapter.jsonl \
  --max-steps 500

2026-09-01 07:12:10,423 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
2026-09-01 07:12:10,848 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-09-01 07:12:10,849 | INFO | datasets | JAX version 0.11.1 available.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 643.24it/s]
Adding EOS to train dataset: 100% 1000/1000 [00:00<00:00, 27004.10 examples/s]
Tokenizing train dataset: 100% 1000/1000 [00:00<00:00, 1757.29 examples/s]
Building labels for train dataset: 100% 1000/1000 [00:00<00:00, 6718.25 examples/s]
Truncating train dataset: 100% 1000/1000 [00:00<00:00, 9595.95 examples/s]
Dropping fully masked examples from train dataset: 100% 1000/1000 [00:00<00:00, 47104.25 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated

In [26]:
!python -m src.qlora.train_adapter_C \
  --dataset /content/banking-compliance-llm-assignment-1a/data/instruction_dataset/train_for_adapter.jsonl \
  --max-steps 500

2026-09-01 07:23:27,102 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
2026-09-01 07:23:27,528 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-09-01 07:23:27,529 | INFO | datasets | JAX version 0.11.1 available.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 651.45it/s]
Adding EOS to train dataset: 100% 1000/1000 [00:00<00:00, 44289.03 examples/s]
Tokenizing train dataset: 100% 1000/1000 [00:00<00:00, 1817.41 examples/s]
Building labels for train dataset: 100% 1000/1000 [00:00<00:00, 6626.97 examples/s]
Truncating train dataset: 100% 1000/1000 [00:00<00:00, 9551.16 examples/s]
Dropping fully masked examples from train dataset: 100% 1000/1000 [00:00<00:00, 51359.87 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated

### 5. Compare adapters

In [31]:
!python -m src.evaluation.adapter_comparison
!python -m src.evaluation.build_report

2026-09-01 07:45:39,353 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 1090.62it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1101.64it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1169.78it/s]
Loading weights: 100% 290/290 [00:00<00:00, 949.25it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1147.27it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1205.90it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1160.18it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1218.38it/s]
Loading weights: 100% 290/290 [00:00<00:00, 1203.94it/s]
2026-09-01 07:46:38,979 | INFO | adapter_comparison | Saved adapter comparison to /content/banking-compliance-llm-assignment-1a/results/adapter_comparison (winner uses transparent keyword-grounding proxy; review for final submission)


In [33]:
import pandas as pd
pd.read_csv("results/adapter_comparison/adapter_comparison.csv")

,prompt,adapter_A,adapter_B,adapter_C,keyword_grounding_scores,most_domain_relevant_by_keyword_proxy
0,What is KYC?,KYC stands for Know Your Customer. It is a pro...,KYC stands for Know Your Customer. It is a pro...,KYC stands for Know Your Customer. It is a reg...,"{""A"": 3, ""B"": 2, ""C"": 3}",A
1,What is Basel III?,Basel III is a regulatory framework that aims ...,Basel III is a regulation that was introduced ...,Basel III is a regulatory framework that was i...,"{""A"": 2, ""B"": 1, ""C"": 1}",A
2,What is RBI LTV Ratio?,RBI LTV ratio is the ratio of the total loan a...,RBI LTV Ratio is the ratio of the total loan a...,RBI LTV Ratio is the ratio of the total loan a...,"{""A"": 4, ""B"": 4, ""C"": 4}",A


### Save Part B outputs to Google Drive

In [34]:
SAVE_DIR = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_PART_B_RESULTS")

items = [
    "data/instruction_dataset",
    "reports",
    "results",
    "models/adapter_A",
    "models/adapter_B",
    "models/adapter_C",
]

for item in items:
    source = PROJECT / item
    destination = SAVE_DIR / item

    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

print("Part B saved to:", SAVE_DIR)

Part B saved to: /content/drive/MyDrive/LLM_ASSIGNMENT_PART_B_RESULTS
